In [1]:
!pip install langgraph langchain-openai fastapi uvicorn python-dotenv aiosqlite sqlalchemy pydantic streamlit --quiet

print("All required packages installed successfully!")

All required packages installed successfully!


In [2]:
!pip install langgraph pandas

In [3]:
!pip install nest_asyncio

In [11]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, List

In [12]:
class AgentState(TypedDict):
    resume_text: str
    ats_score: int
    stage: str
    interview_qa: List[dict]
    screening: dict
    meeting_link: str

In [13]:
def ats_agent(state):
    resume = state["resume_text"]
    
    score = 0
    if "Python" in resume:
        score += 40
    if "FastAPI" in resume:
        score += 40
    if "SQL" in resume:
        score += 20
    
    state["ats_score"] = score
    state["stage"] = "interview" if score >= 80 else "rejected"
    
    return state

In [14]:
def interview_agent(state):
    questions = [
        "What is Python?",
        "Explain FastAPI",
        "What is API?"
    ]
    
    qa = []
    
    for q in questions:
        answer = "demo answer"
        score = 8
        
        qa.append({
            "question": q,
            "answer": answer,
            "score": score
        })
    
    state["interview_qa"] = qa
    state["stage"] = "screening"
    
    return state

In [15]:
def screening_agent(state):
    state["screening"] = {
        "notice_period": "30 days",
        "joining": "Immediate"
    }
    
    state["stage"] = "scheduled"
    
    return state

In [16]:
def scheduling_agent(state):
    state["meeting_link"] = "https://meet.google.com/demo"
    return state

In [17]:
graph = StateGraph(AgentState)

graph.add_node("ats", ats_agent)
graph.add_node("interview", interview_agent)
graph.add_node("screening", screening_agent)
graph.add_node("schedule", scheduling_agent)

graph.set_entry_point("ats")

graph.add_edge("ats", "interview")
graph.add_edge("interview", "screening")
graph.add_edge("screening", "schedule")
graph.add_edge("schedule", END)

recruitment_graph = graph.compile()

In [18]:
result = recruitment_graph.invoke({
    "resume_text": "I know Python FastAPI SQL"
})

print(result)

{'resume_text': 'I know Python FastAPI SQL', 'ats_score': 100, 'stage': 'scheduled', 'interview_qa': [{'question': 'What is Python?', 'answer': 'demo answer', 'score': 8}, {'question': 'Explain FastAPI', 'answer': 'demo answer', 'score': 8}, {'question': 'What is API?', 'answer': 'demo answer', 'score': 8}], 'screening': {'notice_period': '30 days', 'joining': 'Immediate'}, 'meeting_link': 'https://meet.google.com/demo'}


In [19]:
import pandas as pd

db = pd.DataFrame(columns=[
    "resume", "ats_score", "stage"
])

In [20]:
db.loc[len(db)] = [
    result["resume_text"],
    result["ats_score"],
    result["stage"]
]

db

,resume,ats_score,stage
0,I know Python FastAPI SQL,100,scheduled


In [21]:
print("HR DASHBOARD")
print(db)

HR DASHBOARD
                      resume  ats_score      stage
0  I know Python FastAPI SQL        100  scheduled


In [22]:
# Filter shortlisted
db[db["stage"] == "scheduled"]

,resume,ats_score,stage
0,I know Python FastAPI SQL,100,scheduled


In [23]:
def hr_chatbot(query):
    
    if "all candidates" in query:
        return db
    
    if "selected" in query:
        return db[db["stage"] == "scheduled"]
    
    if "rejected" in query:
        return db[db["stage"] == "rejected"]
    
    return "No data found"

In [24]:
hr_chatbot("show all candidates")

,resume,ats_score,stage
0,I know Python FastAPI SQL,100,scheduled


In [25]:
def role_summary():
    return {
        "total_candidates": len(db),
        "selected": len(db[db["stage"] == "scheduled"]),
        "rejected": len(db[db["stage"] == "rejected"])
    }

role_summary()

{'total_candidates': 1, 'selected': 1, 'rejected': 0}

In [26]:
def full_demo():
    resume = "I know Python FastAPI SQL"
    
    result = recruitment_graph.invoke({
        "resume_text": resume
    })
    
    print("Pipeline Output:", result)
    
    # Save
    db.loc[len(db)] = [
        result["resume_text"],
        result["ats_score"],
        result["stage"]
    ]
    
    print("\nDashboard:")
    print(db)
    
    print("\nChatbot Query:")
    print(hr_chatbot("show all candidates"))

In [27]:
full_demo()

Pipeline Output: {'resume_text': 'I know Python FastAPI SQL', 'ats_score': 100, 'stage': 'scheduled', 'interview_qa': [{'question': 'What is Python?', 'answer': 'demo answer', 'score': 8}, {'question': 'Explain FastAPI', 'answer': 'demo answer', 'score': 8}, {'question': 'What is API?', 'answer': 'demo answer', 'score': 8}], 'screening': {'notice_period': '30 days', 'joining': 'Immediate'}, 'meeting_link': 'https://meet.google.com/demo'}

Dashboard:
                      resume  ats_score      stage
0  I know Python FastAPI SQL        100  scheduled
1  I know Python FastAPI SQL        100  scheduled

Chatbot Query:
                      resume  ats_score      stage
0  I know Python FastAPI SQL        100  scheduled
1  I know Python FastAPI SQL        100  scheduled


In [28]:
db

,resume,ats_score,stage
0,I know Python FastAPI SQL,100,scheduled
1,I know Python FastAPI SQL,100,scheduled


In [29]:
hr_chatbot("show all candidates")

,resume,ats_score,stage
0,I know Python FastAPI SQL,100,scheduled
1,I know Python FastAPI SQL,100,scheduled
